# Lesson 08 · Adjunctions

Two functors pointing opposite ways, and a perfect exchange of arrows between them: the pattern behind free constructions, function types, quantifiers and limits.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-08-adjunctions.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §7 — Adjunctions by hand

**`galois.py`** — the sandwich, verified

In [1]:
import math

reals = [i / 7 for i in range(-30, 31)]
ints  = range(-6, 7)
checks = 0
for r in reals:
    for n in ints:
        assert (math.ceil(r) <= n) == (r <= n)    # ceiling  -|  inclusion
        assert (n <= math.floor(r)) == (n <= r)   # inclusion -|  floor
        checks += 2
print(f"ceiling -| inclusion -| floor: verified on {checks} instances")

ceiling -| inclusion -| floor: verified on 1586 instances


**`free.py`** — foldMap is the transpose

In [2]:
def fold_map(f, unit, op):
    "the unique monoid hom  List X -> M  extending  f : X -> M"
    def hat(xs):
        acc = unit
        for x in xs:
            acc = op(acc, f(x))
        return acc
    return hat

f   = {'a': 2, 'b': 3}.get
hat = fold_map(f, 0, lambda m, n: m + n)      # target monoid (Z, +, 0)

import itertools
samples = [list(t) for k in range(4)
                   for t in itertools.product('ab', repeat=k)]
assert hat([]) == 0                            # unit to unit
assert all(hat(xs + ys) == hat(xs) + hat(ys)   # concat to +
           for xs in samples for ys in samples)
assert all(hat([x]) == f(x) for x in 'ab')     # hat . eta == f
print("fold_map is a monoid hom; hat.eta == f;",
      "hat(['a','b','a']) =", hat(['a','b','a']))

fold_map is a monoid hom; hat.eta == f; hat(['a','b','a']) = 7


**`setup.py`** — the category class and two orders

In [3]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

def arrows_between(C, X, Y):
    "all arrows X -> Y of the finite category C"
    return [a for a, (d, c) in C.ar.items() if d == X and c == Y]

def make_poset(objs, leq):
    "a finite preorder as a thin category; the arrow a -> b is named 'a<=b'"
    ar = {f"{a}<={b}": (a, b) for a in objs for b in objs if leq(a, b)}
    comp = {(g, f): f"{ar[f][0]}<={ar[g][1]}"
            for g in ar for f in ar if ar[f][1] == ar[g][0]}
    return FiniteCategory(objs, ar, comp, {a: f"{a}<={a}" for a in objs})

from itertools import product as cartesian

# two small orders, as thin categories
Div   = make_poset({1, 2, 3, 6}, lambda a, b: b % a == 0)  # divisibility
Chain = make_poset({0, 1, 2},    lambda a, b: a <= b)      # 0 <= 1 <= 2

**`adjoint_finder.py`** — search for a partner

In [4]:
def monotone_maps(C, D):
    "all monotone maps C -> D between finite posets"
    Cobjs = sorted(C.obj, key=str)
    for values in cartesian(sorted(D.obj, key=str), repeat=len(Cobjs)):
        g = dict(zip(Cobjs, values))
        if all(not arrows_between(C, x, y) or arrows_between(D, g[x], g[y])
               for x in C.obj for y in C.obj):
            yield g

def find_right_adjoints(f, C, D):
    "all monotone g : D -> C  with   f(x) <= y   iff   x <= g(y)"
    return [g for g in monotone_maps(D, C)
            if all(bool(arrows_between(D, f[x], y)) ==
                   bool(arrows_between(C, x, g[y]))
                   for x in C.obj for y in D.obj)]

f = {0: 1, 1: 2, 2: 6}              # Chain {0,1,2}  ->  Div {1,2,3,6}
print("right adjoints of f    :", find_right_adjoints(f, Chain, Div))

f_bad = {1: 1, 2: 1, 3: 1, 6: 1}    # Div -> Chain, constant at 1
print("right adjoints of f_bad:", find_right_adjoints(f_bad, Div, Chain))

right adjoints of f    : [{1: 0, 2: 1, 3: 0, 6: 2}]
right adjoints of f_bad: []


## Exercise 8

**`census.py`**

In [5]:
from itertools import product as cartesian

X, A, Y = range(2), range(2), range(3)
pairs = [(x, a) for x in X for a in A]
hs = [dict(zip(pairs, vs)) for vs in cartesian(Y, repeat=len(pairs))]

curry  = lambda h: tuple(tuple(h[(x, a)] for a in A) for x in X)
images = {curry(h) for h in hs}

print("|Hom(XxA,Y)| =", len(hs), " distinct curried =", len(images),
      " (|Y|^|A|)^|X| =", (len(Y) ** len(A)) ** len(X))

|Hom(XxA,Y)| = 81  distinct curried = 81  (|Y|^|A|)^|X| = 81


## Exercise 9

**`reverse.py`**

In [6]:
def find_left_adjoints(g, D, C):
    "all monotone f : C -> D  with   f(x) <= y   iff   x <= g(y)"
    return [f for f in monotone_maps(C, D)
            if all(bool(arrows_between(D, f[x], y)) ==
                   bool(arrows_between(C, x, g[y]))
                   for x in C.obj for y in D.obj)]

g = {1: 0, 2: 1, 3: 0, 6: 2}
print(find_left_adjoints(g, Div, Chain))

[{0: 1, 1: 2, 2: 6}]


## Exercise 10

**`quantifiers.py`**

In [7]:
from itertools import combinations

A = {1, 2, 3}; B = {'a', 'b'}
fpt = {1: 'a', 2: 'a', 3: 'b'}
f = lambda x: fpt[x]

def subsets(S):
    S = list(S)
    return [set(c) for r in range(len(S)+1) for c in combinations(S, r)]

E   = lambda S: {f(x) for x in S}                     # ∃f : direct image
pre = lambda T: {x for x in A if f(x) in T}           # f* : preimage
All = lambda S: {b for b in B                         # ∀f
                 if all(x in S for x in A if f(x) == b)}

n = 0
for S in subsets(A):
    for T in subsets(B):
        assert (E(S) <= T) == (S <= pre(T))     # ∃f  -|  f*
        assert (pre(T) <= S) == (T <= All(S))   # f*  -|  ∀f
        n += 2
print(f"quantifier sandwich verified on {n} biconditionals")

quantifier sandwich verified on 64 biconditionals
